# Returning allergen info

Here we want to try to look at inputs being a select list of allergens from a list. So lets say we have bronopol - https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php

We select 'Bronopol' from our list but the ingredients taken from OCR are instead: '2-Bromo-2-nitropropane-1,3-diol' (i.e. an alternative name)

How do we now make sure that when we select 'Bronopol' that we then match to all possible versions of it's name?

1. Select 'Bronopol' on multi-select
2. Return the entire list it belongs to (because these have all the alternative names)
3. Display 'Allergens detected:' then top level allergen (Bronopol) and subset under that as bullet point we'd have '2-Bromo-2-nitropropane-1,3-diol'

## Load packages and check working directory

In [2]:
import easyocr
import cv2
from matplotlib import pyplot as plt
import re
import os
import json

current_dir = os.path.abspath('.')
parent_dir = os.path.dirname(current_dir)
print(f"Current directory: {current_dir}")
print(f"Parent directory: {parent_dir}")

Current directory: c:\Users\SamHenderson-Palmer\Documents\Github\00.Personal\ocr-ingredients-checker\notebooks
Parent directory: c:\Users\SamHenderson-Palmer\Documents\Github\00.Personal\ocr-ingredients-checker


## Set fixed paths and variables

In [3]:
MODEL_PATH = f'{parent_dir}/models/easyocr'
IMAGE_PATH = f'{parent_dir}/assets/sample_images/qv_gentlewash.jpg'
ALLERGEN_LIST_PATH = f'{parent_dir}/assets/data/allergens_cleaned.json'
DELIMITERS = {",", ";", ":"}

with open(ALLERGEN_LIST_PATH, "r", encoding="utf-8") as file:
    ALLERGEN_LIST = json.load(file)

In [4]:
allergen_names = [item["item_name"] for item in ALLERGEN_LIST]
allergen_names

['(nitrobutyl) morpholine / (ethylnitro-trimethylene) dimorpholi',
 '1,2-benzisothiazoline-3-one, sodium salt',
 '1,3-butandiol-dimethacrylate',
 '1,3-diphenylguanidine',
 '1,3,5-tris-(2-hydroxyethyl)-hexahydrotriazine (grotan bk)',
 '1,4-butandioldimethacrylat (budma)',
 '1,4-butanedioldiglycidyl ether',
 '1,6-hexanediolediglycidyl ether',
 '2-(2-aminoethoxy)-eth',
 '2-bromo-2nitropropane-1, 3-diol (bronopol)',
 '2-ethylhexyl acrylate',
 '2-ethylhexyl-4-dimethylaminobenzoate',
 '2-ethylhexyl-p-methoxycinnamate',
 '2-ethylhexyl-p-methoxycinnamate (octinoxate)',
 '2-hydroxy-4-methoxy-benzophenone',
 '2-hydroxy-ethylacrylate',
 '2-hydroxymethyl-2-nitro-1,3-propanediol',
 '2-hydroxypropyl-methacrylate',
 '2-mercaptobenzimidazole',
 '2-mercaptobenzothiazole',
 '3-(4-methylbenzylidene) camphor',
 '3-aminophenol',
 '3,4,4-triclocarban',
 '4-aminoazobenzene',
 '4-chloro-3-cresol (pcmc)',
 '4-chloro-3,5-xylenol (pcmx)',
 '4-hexyl-resorcinol',
 '4-phenylenediamine base',
 '4-tert-butyl-4-methox

In [5]:
selected_allergens = ['2-bromo-2nitropropane-1, 3-diol (bronopol)', 'butylacrylate']

We can return the list of alternative names that corresponds to our selected allergen

In [6]:
data = [item for item in ALLERGEN_LIST if item["item_name"] in selected_allergens]
data

[{'item_name': '2-bromo-2nitropropane-1, 3-diol (bronopol)',
  'url': 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
  'allergens': ['2-Bromo-2-nitropropane-1,3-diol',
   '2-Nitro-2-bromo-1,3-propanediol',
   'Bronidiol',
   'Bronocot',
   'Bronopol',
   'Bronosol',
   'Bronotak',
   'HSDB 7195',
   'Lexgard bronopol',
   'NSC 141021',
   'Onyxide 500',
   'beta-Bromo-beta-nitrotrimethyleneglycol Germall® 11']},
 {'item_name': 'butylacrylate',
  'url': 'https://www.contactdermatitisinstitute.com/butylacrylate.php',
  'allergens': ['2-Propenoic acid, butyl ester',
   '4-02-00-01463 (Beilstein HandbookReference)',
   'AI3-15739',
   'Acrylic acid n-butyl ester',
   'Acrylic acid, butyl ester',
   'BRN 1749970',
   'Butyl 2-propenoate',
   'Butylester kyseliny akrylove',
   'Butylester kyseliny akrylove [Czech]',
   'CCRIS 3401',
   'EINECS 205-480-7',
   'HSDB 305',
   'NSC 5163',
   'n-Butyl acrylate',
   'n-Butyl propenoate']}]

We now need to normalise the ingredients list of each so it can be checked against normalised ingredients list from OCR

In [7]:
for item in data:
    item["allergens"] = [a.lower().strip() for a in item["allergens"]]

In [8]:
data

[{'item_name': '2-bromo-2nitropropane-1, 3-diol (bronopol)',
  'url': 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
  'allergens': ['2-bromo-2-nitropropane-1,3-diol',
   '2-nitro-2-bromo-1,3-propanediol',
   'bronidiol',
   'bronocot',
   'bronopol',
   'bronosol',
   'bronotak',
   'hsdb 7195',
   'lexgard bronopol',
   'nsc 141021',
   'onyxide 500',
   'beta-bromo-beta-nitrotrimethyleneglycol germall® 11']},
 {'item_name': 'butylacrylate',
  'url': 'https://www.contactdermatitisinstitute.com/butylacrylate.php',
  'allergens': ['2-propenoic acid, butyl ester',
   '4-02-00-01463 (beilstein handbookreference)',
   'ai3-15739',
   'acrylic acid n-butyl ester',
   'acrylic acid, butyl ester',
   'brn 1749970',
   'butyl 2-propenoate',
   'butylester kyseliny akrylove',
   'butylester kyseliny akrylove [czech]',
   'ccris 3401',
   'einecs 205-480-7',
   'hsdb 305',
   'nsc 5163',
   'n-butyl acrylate',
   'n-butyl propenoate']}]

Now we need to work out a way of checking for matches. We could experiment by looking at what if 'bronidiol' is returned in OCR ingredients and we selected '2-bromo-2nitropropane-1, 3-diol (bronopol)' as our allergen. What we want is our same list above but only returning the matches.

In [9]:
ocr_ingredients = ['bronidiol']

In [10]:
filtered_list = []

In [11]:
for item in data:
    matched_allergens = [allergen for allergen in item["allergens"] if allergen in ocr_ingredients]

    if matched_allergens:
        filtered_list.append({
            "item_name": item["item_name"],
            "url": item["url"],
            "allergens": matched_allergens
        })

In [12]:
filtered_list

[{'item_name': '2-bromo-2nitropropane-1, 3-diol (bronopol)',
  'url': 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
  'allergens': ['bronidiol']}]